In [1]:
import sys
from pathlib import Path

RETRIEVAL_PATH = Path("..") / "Retrieval"
sys.path.append(str(RETRIEVAL_PATH.resolve()))

In [2]:
import sys, os, csv
import torch, torchaudio
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dotenv import load_dotenv
from Classes.TextAdaptationModule import TextAdaptationModule
from Classes.DataRetrieval import DataRetrieval
from Classes.OpenAIClient import OpenAIClient
from Classes.GeminiClient import GeminiClient
from Classes.InstructionAnalysisModule import InstructionAnalysisModule
from IPython.display import Audio

import numpy as np
import soundfile as sf

sys.path.append(os.path.abspath("../Retrieval"))
sys.path.append(os.path.abspath("../CosyVoice"))
sys.path.append(os.path.abspath("../CosyVoice/third_party/Matcha-TTS"))

try:
    from modelscope import snapshot_download
    from cosyvoice.cli.cosyvoice import CosyVoice2
    from cosyvoice.utils.file_utils import load_wav
    
    model_path = snapshot_download("iic/CosyVoice2-0.5B")
    
except Exception as e:
    raise RuntimeError("Could not import CosyVoice2 / load_wav. Fix your CosyVoice install or imports.") from e

c:\Users\Admin\miniconda3\envs\cosyvoice\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-08-29 02:09:51,337 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


failed to import ttsfrd, use wetext instead


2025-08-29 02:09:52,743 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/iic/CosyVoice2-0.5B/revisions HTTP/1.1" 200 None
2025-08-29 02:09:53,138 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/iic/CosyVoice2-0.5B/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
2025-08-29 02:09:53,140 - modelscope - INFO - Creating symbolic link C:\Users\Admin\.cache\modelscope\hub\iic\iic/CosyVoice2-0___5B -> C:\Users\Admin\.cache\modelscope\hub\iic/CosyVoice2-0.5B.
2025-08-29 02:09:53,141 - modelscope - WARNING - Failed to create symbolic link C:\Users\Admin\.cache\modelscope\hub\iic\iic/CosyVoice2-0___5B -> C:\Users\Admin\.cache\modelscope\hub\iic/CosyVoice2-0.5B: [WinError 1314] A required privilege is not held by the client: 'C:\\Users\\Admin\\.cache\\modelscope\\hub\\iic\\iic\\CosyVoice2-0___5B' -> 'C:\\Users\\Admin\\.cache\\modelscope\\hub\\iic/CosyVoice2-0.5B'


In [3]:
def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def to_float32(audio: Any) -> Tuple[np.ndarray, int]:
    if isinstance(audio, tuple) and len(audio) == 2:
        arr, sr = audio
    elif isinstance(audio, dict) and "audio" in audio and "sample_rate" in audio:
        arr, sr = audio["audio"], audio["sample_rate"]
    else:
        arr, sr = audio, 16000

    arr = np.asarray(arr)
    try:
        arr = arr.detach().cpu().numpy()
    except Exception:
        pass

    if arr.dtype != np.float32:
        arr = arr.astype(np.float32)

    if arr.ndim > 1:
        if arr.shape[0] < arr.shape[-1]:
            arr = arr.T
        arr = arr.mean(axis=1)
    return arr, int(sr)

def _peak_normalize_float32(arr: np.ndarray, peak_db: float = -3.0) -> np.ndarray:
    if arr is None or arr.size == 0:
        return arr
    peak = float(np.max(np.abs(arr)))
    if peak <= 0.0:
        return arr
    target = 10 ** (peak_db / 20.0)
    return arr * (target / peak)

def save_audio_anything(chunks, out_path, default_sr=24000):
    """
    Save CosyVoice outputs into one WAV at the correct sample rate.

    Detect audio keys: audio/wav/pcm/tts_speech/speech/samples
    Detect SR keys: sample_rate/sr/tts_sr/sampleRate/sample_rate_hz/sampling_rate/rate/fs/hz
    Falls back to default_sr (pass DEFAULT_TTS_SR from the model).
    """
    if not chunks:
        return None, "No chunks returned from TTS call"

    KEY_CANDIDATES = ["audio", "wav", "pcm", "tts_speech", "speech", "samples"]
    SR_CANDIDATES  = ["sample_rate", "sr", "tts_sr", "sampleRate",
                      "sample_rate_hz", "sampling_rate", "rate", "fs", "hz"]

    audios = []
    detected_sr = None

    for ch in chunks:
        arr, sr = None, None

        if isinstance(ch, dict):
            for k in KEY_CANDIDATES:
                if k in ch and ch[k] is not None:
                    arr = ch[k]
                    break
            for k in SR_CANDIDATES:
                if k in ch and ch[k] is not None:
                    try:
                        sr = int(ch[k])
                    except Exception:
                        pass
                    break

        if arr is None:
            if isinstance(ch, tuple) and len(ch) == 2:
                arr, sr = ch  # (audio, sr)
            else:
                arr = ch

        try:
            a, s = to_float32({"audio": arr, "sample_rate": sr if sr else default_sr})
            if a is None or a.ndim == 0 or a.size == 0:
                continue
            audios.append(a)
            if sr:
                detected_sr = s
        except Exception:
            continue

    if not audios:
        return None, "No valid audio arrays found in chunks"

    final_sr = int(detected_sr if detected_sr else default_sr)
    concat = np.concatenate(audios, axis=0)
    concat = _peak_normalize_float32(concat, peak_db=-3.0)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(out_path), concat, final_sr)
    return out_path, None


In [4]:
print("[INFO] Loading CosyVoice model...")
cosyvoice = CosyVoice2(model_path, load_jit=False, load_trt=False, load_vllm=False, fp16=False)
print("[INFO] CosyVoice ready.")

DEFAULT_TTS_SR = (
    getattr(cosyvoice, "tts_sample_rate", None)
    or getattr(cosyvoice, "sample_rate", None)
    or 24000   # fallback
)
print("DEFAULT_TTS_SR =", DEFAULT_TTS_SR)

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in .env")

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY not found in .env")

[INFO] Loading CosyVoice model...


c:\Users\Admin\miniconda3\envs\cosyvoice\lib\site-packages\diffusers\models\lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-08-29 02:09:56,001 INFO input frame rate=25
c:\Users\Admin\miniconda3\envs\cosyvoice\lib\site-packages\torch\nn\utils\weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
c:\Users\Admin\miniconda3\envs\cos

2025-08-29 02:09:59,233 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 205
2025-08-29 02:09:59,654 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
2025-08-29 02:09:59,776 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


2025-08-29 02:10:01,182 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 205
2025-08-29 02:10:01,601 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None


[INFO] CosyVoice ready.
DEFAULT_TTS_SR = 24000


In [5]:

import pandas as pd
import json
from pathlib import Path
from Classes.DataRetrieval import DataRetrieval

# File paths
INPUT_JSON = Path("./2_adapted_text_generation/1_prompts_with_speaker_info_noimplicit_final.json")
ACCENT_POOL_TSV = Path("./0_data_with_wer_mos/merged_selected_metadata_wer_mos.tsv")
OUT_DIR = Path("3_Zeroshot_B")
METADATA_PATH = ACCENT_POOL_TSV

# Load accent pool
if not ACCENT_POOL_TSV.exists():
    raise FileNotFoundError(f"Accent TSV not found: {ACCENT_POOL_TSV}")
accent_df = pd.read_csv(ACCENT_POOL_TSV, sep="\t")
ACCENT_POOL_RETRIEVER = DataRetrieval(metadata_path=METADATA_PATH)

# Load JSON
if not INPUT_JSON.exists():
    raise FileNotFoundError(f"JSON not found at {INPUT_JSON}. Please check path.")

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

# Build rows
rows = []
for scen_idx, scen in enumerate(data, start=1):
    sentence = (scen.get("standard_sentence") or "").strip()
    for adapt_idx, adapt in enumerate(scen.get("adaptations") or [], start=1):
        speaker_info = adapt.get("results", {}).get("inferred_speaker_info") or {}
        instruction = adapt.get("instruction") or adapt.get("explicit_instruction") or ""
        uid = adapt.get("uid") or f"S{scen_idx:02d}_A{adapt_idx:02d}"
        rows.append({
            "uid": uid,
            "standard_sentence": sentence,
            "speaker_info": speaker_info,
            "instruction": instruction.strip(),  
        })


print(f"[INFO] Loaded {len(rows)} rows from {INPUT_JSON}")

def _pick_accent_ref(metadata: dict, text: str | None = None) -> dict | None:
    try:
        query = dict(metadata or {})
        if text:
            query["text"] = text
        # print("[DEBUG] Query sent to retriever:", query)
        df = ACCENT_POOL_RETRIEVER.find_relevant(query, top_n=1)  
        if df.empty:
            return None
        ref = df.iloc[0].to_dict()
        ref["audio_path"] = "../" + ref.get("filepath", "")
        return ref
    except Exception as e:
        print("[WARN] Accent pool fallback failed:", repr(e))
        return None


[INFO] Loaded 3600 rows from 2_adapted_text_generation\1_prompts_with_speaker_info_noimplicit_final.json


In [6]:
import numpy as np
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display
import os, csv, traceback

INPUT_SAMPLE_RATE = 16000
OUTPUT_SAMPLE_RATE = 24000

MANIFEST_PATH = OUT_DIR / "Zeroshot_B.tsv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(MANIFEST_PATH, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f, delimiter="\t",
        fieldnames=["uid", "standard_sentence", "instruction", "ref_audio", "ref_transcript", "out_path", "error"]
    )
    writer.writeheader()

    for i, row in enumerate(rows, start=1):
        uid = row.get("uid") or f"row{i:04d}"
        sentence = row.get("standard_sentence", "").strip()
        speaker_info = row.get("speaker_info") or {}
        instruction = row.get("instruction", "").strip()

        print(f"\n=== {i:04d}/{len(rows)} {uid} ===")
        print("Instruction        :", instruction)
        print("Sentence           :", sentence)

        err_msg = ""
        saved_path = ""
        ref_audio = ""
        ref_transcript = ""

        try:
            #  Pick accent-pool reference 
            ref = _pick_accent_ref(speaker_info, sentence)
            if not ref:
                raise RuntimeError("No suitable accent-pool reference found.")
            ref_audio = ref["audio_path"]
            ref_transcript = (ref.get("transcript") or "").strip()
            print("Ref audio          :", ref_audio)

            # Load prompt tensor
            prompt_tensor = load_wav(ref_audio, INPUT_SAMPLE_RATE)

            # Register speaker and run zeroshot generation 
            cosyvoice.add_zero_shot_spk(ref_transcript, prompt_tensor, uid)

            chunks = list(cosyvoice.inference_zero_shot(
                sentence, "", "", zero_shot_spk_id=uid, stream=False
            ))
            if not chunks:
                raise RuntimeError("No chunks returned from inference.")

            # Extract scenario and adaptation numbers from UID
            # UID = S01_A01 -> scen_idx = 1, adapt_idx = 1
            scen_idx = int(uid.split("_")[0][1:])   # "S01" = 1
            adapt_idx = int(uid.split("_")[1][1:])  # "A01" = 1

            accent = speaker_info.get("accent", "UNK").upper()
            gender = speaker_info.get("gender", "U").upper()
            age = speaker_info.get("age", "UNK")
            if isinstance(age, list):
                age = f"{age[0]}"  # Take lower bound if range
            elif isinstance(age, (int, float)):
                age = str(int(age))
            else:
                age = str(age).replace(" ", "").replace("[", "").replace("]", "").split(",")[0]  # cleanup

            out_fname = f"{i:04d}_sc{scen_idx:03d}_a{accent}_g{gender}_age{age}_{adapt_idx}.wav"
            out_path = OUT_DIR / out_fname
            saved_path, err = save_audio_anything(chunks, out_path, default_sr=OUTPUT_SAMPLE_RATE)
            if err:
                raise RuntimeError(err)

            print("Saved ->", saved_path)

        except Exception as e:
            err_msg = f"{type(e).__name__}: {e}"
            print("[ERR]", err_msg)
            traceback.print_exc()

        writer.writerow({
            "uid": uid,
            "standard_sentence": sentence,
            "instruction": instruction,
            "ref_audio": ref_audio,
            "ref_transcript": ref_transcript,
            "out_path": str(saved_path) if saved_path else "",
            "error": err_msg,
        })

print(f"\n[MANIFEST] Wrote: {MANIFEST_PATH}")



=== 0001/3600 S01_A01 ===
Instruction        : Use a male voice with a Canadian accent, moderate speed and a polite tone.
Sentence           : "Could I please see the menu?"
Ref audio          : ../data/selected/AERSC2020/CAN/G20113/G20113S3430.wav


  0%|          | 0/1 [00:00<?, ?it/s]2025-08-29 02:10:04,442 INFO synthesis text "Could I please see the menu?"
c:\Users\Admin\miniconda3\envs\cosyvoice\lib\site-packages\transformers\models\qwen2\modeling_qwen2.py:693: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
2025-08-29 02:10:06,718 INFO yield speech len 2.96, rtf 0.7686714868287783
100%|██████████| 1/1 [00:02<00:00,  2.28s/it]


Saved -> 3_Zeroshot_B\0001_sc001_aCAN_gM_age20_1.wav

=== 0002/3600 S01_A02 ===
Instruction        : The text should be spoken in a young female voice with a Canadian accent. The 'eh' at the end should be pronounced with a typical Canadian inflection.
Sentence           : "Could I please see the menu?"
Ref audio          : ../data/selected/AERSC2020/CAN/G00247/G00247S1221.wav


  0%|          | 0/1 [00:00<?, ?it/s]2025-08-29 02:10:07,103 INFO synthesis text "Could I please see the menu?"
2025-08-29 02:10:08,327 INFO yield speech len 1.48, rtf 0.827172156926748
100%|██████████| 1/1 [00:01<00:00,  1.23s/it]


Saved -> 3_Zeroshot_B\0002_sc001_aCAN_gF_age20_2.wav

=== 0003/3600 S01_A03 ===
Instruction        : Speak in a youthful female voice with a Canadian English accent.
Sentence           : "Could I please see the menu?"
Ref audio          : ../data/selected/AERSC2020/CAN/G70160/G70160S3426.wav


...

Saved -> 3_Zeroshot_B\3598_sc030_aUSA_gM_age18_118.wav

=== 3599/3600 S30_A119 ===
Instruction        : The speaker is a 31-year-old female from USA, so she should speak in an American accent with a medium pitch and speed.
Sentence           : "Who's teaching our next semester's module?"
Ref audio          : ../data/selected/AERSC2020/USA/G11322/G11322S2340.wav


  0%|          | 0/1 [00:00<?, ?it/s]2025-08-29 15:11:44,384 INFO synthesis text "Who's teaching our next semester's module?"
2025-08-29 15:11:47,155 INFO yield speech len 3.48, rtf 0.7962090530614744
100%|██████████| 1/1 [00:02<00:00,  2.78s/it]


Saved -> 3_Zeroshot_B\3599_sc030_aUSA_gF_age31_119.wav

=== 3600/3600 S30_A120 ===
Instruction        : Speak with a middle-aged, female American accent. Use casual and somewhat informal language.
Sentence           : "Who's teaching our next semester's module?"
Ref audio          : ../data/selected/AERSC2020/USA/G12272/G12272S1254.wav


  0%|          | 0/1 [00:00<?, ?it/s]2025-08-29 15:11:47,515 INFO synthesis text "Who's teaching our next semester's module?"
2025-08-29 15:11:49,851 INFO yield speech len 2.92, rtf 0.7999672465128441
100%|██████████| 1/1 [00:02<00:00,  2.34s/it]

Saved -> 3_Zeroshot_B\3600_sc030_aUSA_gF_age40_120.wav

[MANIFEST] Wrote: 3_Zeroshot_B\Zeroshot_B.tsv


In [7]:


from pathlib import Path
import csv, json

INPUT_JSON = Path("./2_adapted_text_generation/1_prompts_with_speaker_info_noimplicit_final.json")
MANIFEST_PATH = Path("./mass_generation_baseline_zeroshot_framework_prompt/zeroshot_manifest.tsv")

# How many scenarios, target adaptations, and how many are valid (non-[ERROR])?
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

num_scenarios = len(data)
target_total = sum(len(s.get("adaptations") or []) for s in data)

# Count invalids
invalids = []
valid_total = 0
for scen_idx, scen in enumerate(data, start=1):
    txt = (scen.get("standard_sentence") or "").strip()
    if len(txt) >= 2 and txt[0] == txt[-1] and txt[0] in ("'", '"'):
        txt = txt[1:-1]
    for adapt_idx, adapt in enumerate(scen.get("adaptations") or [], start=1):
        instr = (adapt.get("explicit_instruction") or "").strip()
        if not txt or not instr or instr.startswith("[ERROR]"):
            invalids.append((scen_idx, adapt_idx, instr))
        else:
            valid_total += 1

print(f"[SCENARIOS] {num_scenarios}")
print(f"[ADAPTATIONS in JSON] {target_total}")
print(f"[INVALID skipped rows] {len(invalids)} (expected 3)")
print(f"[VALID rows your loop runs] {valid_total}")

# What actually got saved
saved = 0
failed = 0
missing_on_disk = 0

if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, "r", encoding="utf-8", newline="") as f:
        r = csv.DictReader(f, delimiter="\t")
        rows = list(r)

    for row in rows:
        err = (row.get("error") or "").strip()
        outp = (row.get("output_path") or "").strip()
        if err:
            failed += 1
        elif outp:
            if Path(outp).exists():
                saved += 1
            else:
                missing_on_disk += 1

    print(f"[MANIFEST] rows: {len(rows)}")
    print(f"[SAVED files] {saved}")
    print(f"[FAILED (error set)] {failed}")
    print(f"[MISSING on disk despite path in manifest] {missing_on_disk}")
else:
    print(f"[WARN] Manifest not found at {MANIFEST_PATH}")


[SCENARIOS] 30
[ADAPTATIONS in JSON] 3600
[INVALID skipped rows] 0 (expected 3)
[VALID rows your loop runs] 3600
[MANIFEST] rows: 3600
[SAVED files] 0
[FAILED (error set)] 5
[MISSING on disk despite path in manifest] 0
